# Training our own SFace

This notebook trains the face recognizer of the college face recognition project on Kaggle's free GPU.
Before running it:

1. **Add Input** (right-hand panel): add the dataset `https://www.kaggle.com/datasets/yakhyokhuja/webface-112x112`.
2. **Settings → Accelerator**: `GPU T4 x2`. **Settings → Internet**: on (to download LFW and the YuNet model).
3. **Save Version → Save & Run All (Commit)**. It runs in the background, so the browser can be closed.

When it has finished, download `training/runs/mobilefacenet_sface_v2/` from the version's **Output** tab
and put it in the project's `training/runs/` folder.


In [ ]:
!nvidia-smi
import os, shutil, subprocess, time
from pathlib import Path
import cv2, torch

NOTEBOOK_STARTED = time.time()
print("PyTorch", torch.__version__, "| OpenCV", cv2.__version__, "|", torch.cuda.device_count(), "GPU(s)")


def run(command):
    """Run a command, show its output, and stop the notebook if it fails (a plain !command would carry on)."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Failed: {command}")


The project's own files, exactly as they are in the project folder:

In [ ]:
import os
os.makedirs('app', exist_ok=True)
os.makedirs('training', exist_ok=True)


In [ ]:
open("app/__init__.py", "w").close()  # an empty file


In [ ]:
%%writefile app/config.py
"""Every setting of the project in one place."""

from pathlib import Path

PROJECT_DIR = Path(__file__).resolve().parent.parent

PEOPLE_DIR = PROJECT_DIR / "data" / "people"
# The numbers our own SFace gives each registered person.
DATABASE_FILE = PROJECT_DIR / "data" / "face_database_ours.json"
ATTENDANCE_FILE = PROJECT_DIR / "data" / "attendance.csv"
RESULTS_DIR = PROJECT_DIR / "data" / "results"
MODELS_DIR = PROJECT_DIR / "models"

# Step 1, finding faces: OpenCV's pretrained YuNet face detector, from the official OpenCV model zoo.
# It is downloaded automatically on first run.
DETECTOR_MODEL_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
DETECTOR_MODEL_FILE = MODELS_DIR / "face_detection_yunet_2023mar.onnx"
# Step 2, recognizing faces: our own SFace, trained with the code in training/ (see "Our own SFace"
# in CLAUDE.md) and saved by "python -m training.export_onnx". It comes with the project.
OUR_MODEL_FILE = MODELS_DIR / "our_sface.onnx"

# How sure (0 to 1) the detector must be before it reports a face.
DETECTION_CONFIDENCE = 0.8
# Photos bigger than this (longest side, in pixels) are shrunk before searching for faces,
# because the detector misses faces that are too big (like close-up selfies).
# Raise it to 1280 only for large group photos in which every face is tiny.
DETECTION_MAX_SIZE = 640

# The mirror trick: each face is also turned into numbers as its mirror image, and the two results
# are averaged. Measured on LFW (see CLAUDE.md), it helps our model a little when a person is
# registered with 1 photo, and costs about 3 ms per face.
# It changes every embedding, so after changing it, register everyone again from the photo folders.
MIRROR_TRICK = True

# A face only gets a name if its similarity to that person reaches this value. Chosen on the LFW face
# dataset with 100 registered people (see CLAUDE.md) as the lowest value that gives at most 0.4% of
# strangers a name. At 0.48, 96.3% of registered people (5 photos each) were named correctly and 0.3%
# of strangers got a name.
# Lower it if registered people often show up as "Unknown"; raise it if strangers get names.
MATCH_THRESHOLD = 0.48

# 0 is the built-in webcam; try 1 or 2 for an external one.
CAMERA_INDEX = 0
# A person must be recognized in this many video frames in a row to be marked present.
FRAMES_TO_CONFIRM = 10


In [ ]:
%%writefile app/face_engine.py
"""The two AI models that do the real work, and the alignment step between them.

Step 1, detect_faces():  WHERE are the faces in an image?  (OpenCV's pretrained YuNet model)
Step 2, get_embedding(): turn one face into 128 numbers, called an "embedding".
         First align_face() turns the face upright and cuts it out at 112x112 pixels. Then our own
         SFace, trained with the code in training/, turns it into the numbers.
         Photos of the same person give similar numbers, photos of different people don't.
"""

import urllib.request
from contextlib import contextmanager
from pathlib import Path

import cv2
import numpy as np

from app import config

# Where alignment puts the 5 face points in the 112x112 face picture: right eye, left eye, nose tip,
# right and left mouth corner. This is the standard "ArcFace" template. Our model was trained on faces
# aligned to it, and OpenCV's own alignment function (alignCrop) uses the same one.
TEMPLATE_POINTS = np.array(
    [[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366], [41.5493, 92.3655], [70.7299, 92.2041]]
)
FACE_SIZE = 112


def download_model(url: str, path: Path) -> Path:
    """Download a model file the first time it is needed."""
    if not path.exists():
        print(f"Downloading {path.name} (only needed once)...")
        path.parent.mkdir(parents=True, exist_ok=True)
        # Download under a temporary name so an interrupted download is never mistaken for a model.
        temporary_path = path.with_suffix(".download")
        urllib.request.urlretrieve(url, temporary_path)
        temporary_path.replace(path)
    return path


@contextmanager
def opencv_warnings_hidden():
    """OpenCV 5 prints a harmless "Targets are not supported" warning while it loads these models."""
    previous_log_level = cv2.utils.logging.getLogLevel()
    cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)
    try:
        yield
    finally:
        cv2.utils.logging.setLogLevel(previous_log_level)


def similarity_transform(points: np.ndarray, target: np.ndarray) -> np.ndarray:
    """The turn, scale and shift (no stretching) that move `points` as close as possible onto `target`.

    Returned as the 2x3 matrix cv2.warpAffine expects. It is the least-squares fit known as Umeyama's
    method, the same calculation OpenCV's alignCrop does.
    """
    points_mean, target_mean = points.mean(axis=0), target.mean(axis=0)
    points_centred, target_centred = points - points_mean, target - target_mean
    covariance = target_centred.T @ points_centred / len(points)
    u, spread, vt = np.linalg.svd(covariance)
    # Only a turn, never a mirror image: if the closest fit would mirror the face, it is not allowed to.
    signs = np.array([1.0, 1.0 if np.linalg.det(covariance) >= 0 else -1.0])
    turn = u @ np.diag(signs) @ vt
    scale = spread @ signs / points_centred.var(axis=0).sum()
    shift = target_mean - scale * turn @ points_mean
    return np.hstack([scale * turn, shift[:, None]])


def align_face(image: np.ndarray, face: np.ndarray) -> np.ndarray:
    """Cut one detected face out of the image as a 112x112 picture, turned and scaled so that its
    5 points land as close as possible on the template: eyes level, always the same size and place."""
    points = face[4:14].reshape(5, 2).astype(np.float64)
    matrix = similarity_transform(points, TEMPLATE_POINTS)
    return cv2.warpAffine(image, matrix, (FACE_SIZE, FACE_SIZE), flags=cv2.INTER_LINEAR)


def load_network(model_file: Path) -> cv2.dnn.Net:
    """Load a face network saved as ONNX into OpenCV's deep-learning engine, which runs it."""
    with opencv_warnings_hidden():
        return cv2.dnn.readNetFromONNX(str(model_file))


def run_network(network: cv2.dnn.Net, aligned_face: np.ndarray) -> np.ndarray:
    """The 128 numbers a face network gives one aligned face (not yet scaled to length 1).

    Our network takes RGB pixels from 0 to 255 and scales them itself (see training/mobilefacenet.py),
    so the only preparation is turning OpenCV's BGR colour order into RGB.
    """
    network.setInput(cv2.dnn.blobFromImage(aligned_face, swapRB=True))
    return network.forward().flatten()


class FaceEngine:
    def __init__(self, load_recognizer: bool = True):
        """load_recognizer=False loads only the detector: training/datasets.py uses that to align
        the LFW test photos before our model has been trained."""
        detector_file = download_model(config.DETECTOR_MODEL_URL, config.DETECTOR_MODEL_FILE)
        with opencv_warnings_hidden():
            self.detector = cv2.FaceDetectorYN.create(
                str(detector_file), "", (320, 320), config.DETECTION_CONFIDENCE
            )
        self.network = None
        if load_recognizer:
            if not config.OUR_MODEL_FILE.exists():
                raise SystemExit(
                    f"{config.OUR_MODEL_FILE} is missing. It comes with the project (download the project "
                    "again), or make it from the training results with: python -m training.export_onnx"
                )
            self.network = load_network(config.OUR_MODEL_FILE)

    def detect_faces(self, image: np.ndarray) -> list[np.ndarray]:
        """Step 1: find every face in an image.

        Each face is a row of 15 numbers:
            [0:4]   the box around the face: x, y, width, height
            [4:14]  five (x, y) points: right eye, left eye, nose tip, right and left mouth corner
            [14]    how confident the detector is that this is a face (0 to 1)
        """
        height, width = image.shape[:2]
        scale = min(1.0, config.DETECTION_MAX_SIZE / max(height, width))
        small_image = cv2.resize(image, None, fx=scale, fy=scale) if scale < 1.0 else image

        self.detector.setInputSize((small_image.shape[1], small_image.shape[0]))
        _, faces = self.detector.detect(small_image)
        if faces is None:
            return []

        faces[:, :14] /= scale  # boxes and points back to the size of the original image
        return list(faces)

    def get_embedding(self, image: np.ndarray, face: np.ndarray) -> np.ndarray:
        """Step 2: turn one detected face into an embedding (128 numbers, scaled to length 1)."""
        aligned_face = align_face(image, face)
        embedding = run_network(self.network, aligned_face)
        embedding /= np.linalg.norm(embedding)
        if config.MIRROR_TRICK:
            # The mirror image is the same person seen from a slightly different side; averaging
            # the two evens out how the model sees each half of the face.
            mirrored = run_network(self.network, cv2.flip(aligned_face, 1))
            embedding = embedding + mirrored / np.linalg.norm(mirrored)
        return embedding / np.linalg.norm(embedding)


In [ ]:
%%writefile training/__init__.py
"""Training our own SFace. Run the steps from the project folder with  python -m training.<step>
(see "Our own SFace" in CLAUDE.md)."""


In [ ]:
%%writefile training/mobilefacenet.py
"""The network of our own SFace: MobileFaceNet.

It turns one aligned 112x112 face into 128 numbers (the embedding), the same job OpenCV's SFace does.
MobileFaceNet (Chen et al., 2018) is a small network made for phones, with about 1 million weights.
It is the same network as in the official SFace code (github.com/zhongyy/SFace,
SFace_torch/backbone/model_mobilefacenet.py), written out here with explanations.

Input:  a batch of faces, shape (batch, 3, 112, 112), RGB order, pixel values 0 to 255.
        That is exactly what the app gives it (run_network in app/face_engine.py only swaps
        OpenCV's BGR colour order to RGB), the same input OpenCV's own SFace model takes.
Output: shape (batch, 128). Not yet scaled to length 1; the app does that.

How the picture shrinks on its way through:
    112x112 -> 56x56 -> 28x28 -> 14x14 -> 7x7 -> 1x1 (512 numbers) -> 128 numbers
"""

import torch
from torch import nn

EMBEDDING_SIZE = 128


class ConvBlock(nn.Sequential):
    """Convolution, then batch normalization, then PReLU (a ReLU that learns its slope for negative numbers).

    groups=channels makes it a "depthwise" convolution: each channel is filtered on its own,
    which is much cheaper than mixing all channels at once.
    """

    def __init__(self, in_channels, out_channels, kernel=1, stride=1, padding=0, groups=1, activation=True):
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_channels, eps=1e-3),
        ]
        if activation:
            layers.append(nn.PReLU(out_channels))
        super().__init__(*layers)


class Bottleneck(nn.Module):
    """MobileNetV2's building block: widen with a 1x1 convolution, filter each channel with a
    depthwise 3x3 convolution, then narrow again with a 1x1 convolution (no activation).

    With residual=True the block's input is added to its output, so the block only has to learn
    a correction. That makes deep networks much easier to train.
    """

    def __init__(self, in_channels, out_channels, wide_channels, stride=1, residual=False):
        super().__init__()
        self.layers = nn.Sequential(
            ConvBlock(in_channels, wide_channels),
            ConvBlock(wide_channels, wide_channels, kernel=3, stride=stride, padding=1, groups=wide_channels),
            ConvBlock(wide_channels, out_channels, activation=False),
        )
        self.residual = residual

    def forward(self, x):
        return x + self.layers(x) if self.residual else self.layers(x)


def repeated(count, channels, wide_channels):
    return nn.Sequential(*[Bottleneck(channels, channels, wide_channels, residual=True) for _ in range(count)])


class MobileFaceNet(nn.Module):
    def __init__(self, embedding_size=EMBEDDING_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64, kernel=3, stride=2, padding=1),  # 56x56
            ConvBlock(64, 64, kernel=3, padding=1, groups=64),
            Bottleneck(64, 64, 128, stride=2),  # 28x28
            repeated(4, 64, 128),
            Bottleneck(64, 128, 256, stride=2),  # 14x14
            repeated(6, 128, 256),
            Bottleneck(128, 128, 512, stride=2),  # 7x7
            repeated(2, 128, 256),
            ConvBlock(128, 512),
            # "Global depthwise convolution": one 7x7 filter per channel turns the 7x7 map into 1 number.
            # Unlike a plain average, it learns that some parts of the face (eyes, nose) matter more.
            ConvBlock(512, 512, kernel=7, groups=512, activation=False),  # 1x1
            nn.Flatten(),
        )
        self.embedding = nn.Sequential(
            nn.Linear(512, embedding_size),
            nn.BatchNorm1d(embedding_size, eps=2e-5),
        )

    def forward(self, faces: torch.Tensor) -> torch.Tensor:
        # Pixels 0..255 become roughly -1..1, which networks train well on. Doing it here, inside the
        # model, means the exported ONNX file accepts the raw pixels OpenCV gives it.
        x = (faces - 127.5) / 128.0
        return self.embedding(self.features(x))


In [ ]:
%%writefile training/sface_loss.py
"""The SFace loss: what makes the network learn good embeddings.

Paper: Zhong et al., "SFace: Sigmoid-Constrained Hypersphere Loss for Robust Face Recognition",
IEEE Transactions on Image Processing, 2021. Official code: github.com/zhongyy/SFace
(SFace_torch/head/metrics.py, class SFaceLoss). This file does the same calculation.

The idea, step by step:

1. Every person in the training set (10,572 of them) gets a "class centre": a learnable direction
   with 128 numbers, stored in self.centres. Embeddings and centres are both scaled to length 1,
   so they all lie on a sphere (the "hypersphere").
2. For each face we measure the angle between its embedding and every centre (via cosine).
   Good embeddings have a SMALL angle to their own person's centre ("intra-class")
   and LARGE angles to everyone else's centre ("inter-class").
3. The loss pulls the face towards its own centre and pushes it away from the others:
       loss = -s * cos(angle to own centre) * pull_weight
              + sum over other people of  s * cos(angle to their centre) * push_weight
4. SFace's trick is in the two weights, which are sigmoid curves of the angle:
       pull_weight = sigmoid(k * (angle_own - a))      close to 1 while the angle is above a,
                                                       fading to 0 once the face is close enough
       push_weight = sigmoid(-k * (angle_other - b))   close to 1 while another centre is nearer than b,
                                                       fading to 0 once it is far enough away
   So the network stops pulling and pushing once a face is "good enough". Older losses keep
   pushing forever, which makes the network overfit to noisy, mislabelled web photos.
   The weights are computed under torch.no_grad(): they only decide HOW HARD to pull or push,
   and no learning flows through them.

Settings (the official ones): s = 64 (scale), k = 80 (how steep the sigmoids are),
a = 0.87 and b = 1.2 radians (about 50 and 69 degrees), from the official CASIA-WebFace command.
"""

import math

import torch
import torch.nn.functional as F
from torch import nn


class SFaceLoss(nn.Module):
    def __init__(self, embedding_size, num_people, s=64.0, k=80.0, a=0.87, b=1.2):
        super().__init__()
        self.s, self.k, self.a, self.b = s, k, a, b
        self.centres = nn.Parameter(torch.empty(num_people, embedding_size))
        # Same starting values as the official code (normal, standard deviation 2 * sqrt(2 / num_people)).
        nn.init.normal_(self.centres, 0.0, 2.0 * math.sqrt(2.0 / num_people))

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor):
        """Returns (loss, cosines). cosines has one row per face and one column per person."""
        cosines = F.linear(F.normalize(embeddings), F.normalize(self.centres))
        own_cosine = cosines.gather(1, labels[:, None]).squeeze(1)

        with torch.no_grad():
            own_angle = torch.acos(own_cosine.clamp(-1.0, 1.0))
            pull_weight = torch.sigmoid(self.k * (own_angle - self.a))
            other_angles = torch.acos(cosines.clamp(-1.0, 1.0))
            push_weights = torch.sigmoid(-self.k * (other_angles - self.b))
            push_weights.scatter_(1, labels[:, None], 0.0)  # a face is never pushed away from its own person

        pull_loss = -(pull_weight * self.s * own_cosine).mean()
        push_loss = (push_weights * self.s * cosines).sum(dim=1).mean()
        return pull_loss + push_loss, cosines


In [ ]:
%%writefile training/datasets.py
"""Getting the face photos ready for training and testing.  Run:
    python -m training.datasets                        (packs datasets/webface_112x112.zip)
    python -m training.datasets --webface <folder>     (packs a folder of person folders, as on Kaggle)

TRAINING: CASIA-WebFace, already aligned to 112x112 (datasets/webface_112x112.zip, from Kaggle
    "yakhyokhuja/webface-112x112"): 490,623 web photos of 10,572 celebrities, one folder per person.
    pack_webface() copies every photo's bytes into one big file (webface_images.bin) with an index
    (webface_index.npz: where each photo starts, how long it is, and its person number).
    Reading from one big file is much faster than opening 490,623 small ones.

TESTING: LFW (datasets/lfw.tgz and pairs.txt, the figshare copies scikit-learn uses; downloaded here
    if missing). align_lfw() runs the app's own detection (YuNet) and alignment (align_face in
    app/face_engine.py) on every LFW photo, so the tests see faces the way the app does. Saved as
    lfw_aligned.npz.

The training photos were aligned with the same 5-point template as the app's align_face (checked: see
"Our own SFace" in CLAUDE.md), so training faces look like the faces the app will show the model.
"""

import argparse
import hashlib
import os
import tarfile
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np

from app.config import PROJECT_DIR

# On Kaggle this points to /tmp, which is not saved with the notebook's output (see the notebook).
DATASETS_DIR = Path(os.environ.get("FACE_DATASETS_DIR", PROJECT_DIR / "datasets"))
WEBFACE_ZIP = DATASETS_DIR / "webface_112x112.zip"
WEBFACE_IMAGES = DATASETS_DIR / "webface_images.bin"
WEBFACE_INDEX = DATASETS_DIR / "webface_index.npz"
LFW_TGZ = DATASETS_DIR / "lfw.tgz"
LFW_PAIRS = DATASETS_DIR / "pairs.txt"
LFW_ALIGNED = DATASETS_DIR / "lfw_aligned.npz"

# The figshare copies of LFW and their SHA-256 checksums, as published by scikit-learn.
LFW_TGZ_URL = "https://ndownloader.figshare.com/files/5976018"
LFW_PAIRS_URL = "https://ndownloader.figshare.com/files/5976006"
LFW_TGZ_SHA256 = "055f7d9c632d7370e6fb4afc7468d40f970c34a80d4c6f50ffec63f5a8d536c0"
LFW_PAIRS_SHA256 = "ea42330c62c92989f9d7c03237ed5d591365e89b3e649747777b70e692dc1592"


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# ---------------------------------------------------------------- training set


def webface_photos(source: Path):
    """Yields (person folder name, photo bytes) for every photo, from the zip or from a folder."""
    if source.suffix == ".zip":
        with zipfile.ZipFile(source) as archive:
            for entry in archive.infolist():
                if entry.filename.lower().endswith(".jpg"):
                    yield Path(entry.filename).parent.name, archive.read(entry)
        return
    # On network storage like Kaggle's dataset folders, every file takes a while to arrive, so
    # reading them one by one managed fewer than 100 photos a second. 64 threads each wait for
    # their own file at the same time. pool.map hands the results back in the original order.
    with ThreadPoolExecutor(64) as pool:
        person_dirs = sorted(source.iterdir())
        photos = [photo for listing in pool.map(lambda d: sorted(d.glob("*.jpg")), person_dirs) for photo in listing]
        print(f"  found {len(photos):,} photos in {len(person_dirs):,} folders", flush=True)
        for start in range(0, len(photos), 10000):  # 10,000 at a time, so memory use stays small
            chunk = photos[start : start + 10000]
            for photo, data in zip(chunk, pool.map(Path.read_bytes, chunk)):
                yield photo.parent.name, data


def pack_webface(source: Path = WEBFACE_ZIP) -> None:
    DATASETS_DIR.mkdir(parents=True, exist_ok=True)
    offsets, lengths, folders = [], [], []
    position = 0
    started = time.time()
    with open(WEBFACE_IMAGES, "wb") as images_file:
        for number, (folder, data) in enumerate(webface_photos(source)):
            images_file.write(data)
            offsets.append(position)
            lengths.append(len(data))
            folders.append(folder)
            position += len(data)
            if number % 50000 == 0:
                print(f"  packed {number:,} photos ({time.time() - started:.0f} s)", flush=True)

    # Folders are named id_0 ... id_10571; in that order they become person numbers 0 ... 10571 (the labels).
    names = sorted(set(folders), key=lambda name: int(name.split("_")[1]))
    label_of = {name: label for label, name in enumerate(names)}
    labels = np.array([label_of[folder] for folder in folders], np.int32)
    np.savez(WEBFACE_INDEX, offsets=np.array(offsets, np.int64), lengths=np.array(lengths, np.int32), labels=labels)
    print(f"Training set: {len(labels):,} photos of {len(names):,} people, {position / 1e9:.2f} GB")


def check_webface_photos(count: int = 2000) -> None:
    """Check that the packed photos decode and are all 112x112."""
    index = np.load(WEBFACE_INDEX)
    images = np.memmap(WEBFACE_IMAGES, dtype=np.uint8, mode="r")
    sizes = set()
    count = min(count, len(index["labels"]))
    for number in np.random.default_rng(0).choice(len(index["labels"]), count, replace=False):
        start = index["offsets"][number]
        image = cv2.imdecode(np.asarray(images[start : start + index["lengths"][number]]), cv2.IMREAD_COLOR)
        sizes.add(None if image is None else image.shape)
    print(f"Sizes seen in {count} random training photos: {sizes}")


def webcam_style(image: np.ndarray) -> np.ndarray:
    """Randomly makes a clean training photo look more like a webcam picture ("augmentation").

    The training photos are mostly sharp, well-lit celebrity photos; webcams give small, blurry,
    dark or washed-out faces. Each change happens to only some photos, so most stay clean:
      - 20%: low resolution: shrunk to 24-80 pixels wide and enlarged back (a face far from the camera)
      - 20%: blur (out of focus, or moving)
      - 30%: brightness and contrast changed (dark rooms, bright windows)
    """
    if np.random.rand() < 0.2:
        size = np.random.randint(24, 81)
        small = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
        image = cv2.resize(small, (112, 112), interpolation=cv2.INTER_LINEAR)
    if np.random.rand() < 0.2:
        image = cv2.GaussianBlur(image, (0, 0), sigmaX=np.random.uniform(0.5, 1.5))
    if np.random.rand() < 0.3:
        contrast, brightness = np.random.uniform(0.6, 1.4), np.random.uniform(-40, 40)
        image = cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)  # also clips to 0..255
    return image


class WebFaceDataset:
    """The training photos for PyTorch: returns (face, person number).

    face: uint8 tensor (3, 112, 112), RGB, flipped left-right half the time ("augmentation":
    a mirrored face is still the same person, so this doubles the variety for free), and with
    random webcam-style changes (see webcam_style).
    """

    def __init__(self, flip: bool = True, webcam_changes: bool = True):
        index = np.load(WEBFACE_INDEX)
        self.offsets, self.lengths, self.labels = index["offsets"], index["lengths"], index["labels"]
        self.num_people = int(self.labels.max()) + 1
        self.flip = flip
        self.webcam_changes = webcam_changes
        self.images = None  # opened in each worker process, see __getitem__

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, number):
        import torch

        if self.images is None:
            # A memory map lets every worker read the big file without loading all of it into memory.
            self.images = np.memmap(WEBFACE_IMAGES, dtype=np.uint8, mode="r")
        start = self.offsets[number]
        image = cv2.imdecode(np.asarray(self.images[start : start + self.lengths[number]]), cv2.IMREAD_COLOR)
        if self.webcam_changes:
            image = webcam_style(image)
        image = image[:, :, ::-1]  # OpenCV gives BGR; the model is trained on RGB, like OpenCV's SFace
        if self.flip and np.random.rand() < 0.5:
            image = image[:, ::-1]
        return torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1))), int(self.labels[number])


# ---------------------------------------------------------------- test set


def align_lfw() -> None:
    from app.face_engine import FaceEngine, align_face

    for path, url, expected in ((LFW_TGZ, LFW_TGZ_URL, LFW_TGZ_SHA256), (LFW_PAIRS, LFW_PAIRS_URL, LFW_PAIRS_SHA256)):
        if not path.exists():
            print(f"Downloading {path.name}...", flush=True)
            DATASETS_DIR.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, path.with_suffix(".download"))
            path.with_suffix(".download").replace(path)
        if sha256(path) != expected:
            raise SystemExit(f"{path.name} does not match its published checksum; download it again.")
    print("LFW files match their published checksums.")

    engine = FaceEngine(load_recognizer=False)  # only detection: our model doesn't exist before training
    # LFW photos are small (250x250) news photos; accept slightly less certain detections than the app.
    engine.detector.setScoreThreshold(0.5)

    names, stems, crops, found_faces, missed = [], [], [], [], {}
    with tarfile.open(LFW_TGZ) as archive:
        for member in archive:
            if not member.name.lower().endswith(".jpg"):
                continue
            data = np.frombuffer(archive.extractfile(member).read(), np.uint8)
            image = cv2.imdecode(data, cv2.IMREAD_COLOR)
            names.append(Path(member.name).parent.name)
            stems.append(Path(member.name).stem)
            found = engine.detect_faces(image)
            if found:
                # LFW photos are centred on the named person, so take the face closest to the centre.
                centre = np.array(image.shape[1::-1]) / 2
                face = min(found, key=lambda f: np.linalg.norm(f[:2] + f[2:4] / 2 - centre))
                found_faces.append(face)
                crops.append(align_face(image, face))
            else:
                missed[len(crops)] = image
                crops.append(None)
            if len(crops) % 2000 == 0:
                print(f"  {len(crops):,} LFW photos done")

    # Where no face was found, use the average landmark positions of all the other photos instead.
    average_face = np.mean(found_faces, axis=0)
    for number, image in missed.items():
        crops[number] = align_face(image, average_face)

    detected = np.ones(len(crops), bool)
    detected[list(missed)] = False
    np.savez(
        LFW_ALIGNED,
        crops=np.stack(crops),  # (13233, 112, 112, 3), BGR like everything in OpenCV
        names=np.array(names),
        stems=np.array(stems),
        detected=detected,
    )
    print(f"LFW: {len(crops):,} photos aligned; no face found in {len(missed)} (average landmarks used for those).")


def load_lfw_pairs(stems: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The official 6,000 LFW test pairs, as photo numbers.

    Returns (first, second, same): the two photos of each pair and whether they show the same person.
    pairs.txt holds 10 groups ("folds") of 300 same-person pairs followed by 300 different-person pairs.
    """
    number_of = {stem: number for number, stem in enumerate(stems)}
    first, second, same = [], [], []
    for line in LFW_PAIRS.read_text().splitlines()[1:]:
        parts = line.split()
        if len(parts) == 3:
            name1, n1, name2, n2 = parts[0], parts[1], parts[0], parts[2]
        else:
            name1, n1, name2, n2 = parts
        first.append(number_of[f"{name1}_{int(n1):04d}"])
        second.append(number_of[f"{name2}_{int(n2):04d}"])
        same.append(len(parts) == 3)
    return np.array(first), np.array(second), np.array(same)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--webface", type=Path, default=WEBFACE_ZIP, help="the zip, or a folder of person folders")
    args = parser.parse_args()
    if not WEBFACE_INDEX.exists():
        pack_webface(args.webface)
    check_webface_photos()
    if not LFW_ALIGNED.exists():
        align_lfw()


In [ ]:
%%writefile training/evaluate.py
"""Testing our face recognizer on LFW.  Run:
    python -m training.evaluate                     (our own SFace)
    python -m training.evaluate --compare-opencv    (also OpenCV's pretrained SFace, as a yardstick)

Both tests use the LFW faces aligned by the app's own code (training/datasets.py), and our model is
run from its exported ONNX file by OpenCV's deep-learning engine, exactly as the app runs it.
OpenCV's SFace is not part of the app: --compare-opencv downloads it only to measure how close
our model comes to it, tested in exactly the same way.

1. The standard LFW test ("verification"): 6,000 pairs of photos, half of them the same person.
   A pair counts as "same person" when the two embeddings' similarity reaches a threshold. The
   threshold is chosen on 9 of the 10 groups of pairs and tested on the 10th, ten times over, and
   the accuracy is averaged. Research papers report this number, so it compares us with them.

2. The college test ("identification"), as in "Accuracy measured on the LFW dataset" in CLAUDE.md:
   register 100 people with 5 photos each, then check, at each threshold, how often up to 5 other
   photos of each of them get the right name and how often 1,000 strangers get a name. This is what
   the app actually does, and it is how MATCH_THRESHOLD is chosen.
"""

from collections import defaultdict
from pathlib import Path

import numpy as np

from app import config
from app.face_engine import download_model, load_network, run_network
from training.datasets import DATASETS_DIR, LFW_ALIGNED, load_lfw_pairs

# The threshold rule: the lowest threshold at which no more than 0.4% of strangers get a name,
# with 100 registered people.
STRANGERS_ALLOWED = 0.004
# OpenCV's pretrained SFace, only for --compare-opencv: the yardstick our model was measured against.
OPENCV_SFACE_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx"
OPENCV_SFACE_FILE = config.MODELS_DIR / "face_recognition_sface_2021dec.onnx"


def load_lfw():
    data = np.load(LFW_ALIGNED)
    return data["crops"], data["names"], data["stems"]


def normalize(embeddings: np.ndarray) -> np.ndarray:
    return embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)


def embed_with_opencv(model_file: Path, crops: np.ndarray) -> np.ndarray:
    """Embeddings exactly as the app makes them (run_network on each aligned face)."""
    network = load_network(model_file)
    return normalize(np.stack([run_network(network, crop) for crop in crops]))


def embed_with_torch(model, crops: np.ndarray, batch_size: int = 512) -> np.ndarray:
    """The same embeddings straight from PyTorch, much faster on the GPU (used during training)."""
    import torch

    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    parts = []
    with torch.no_grad():
        for start in range(0, len(crops), batch_size):
            rgb = np.ascontiguousarray(crops[start : start + batch_size, :, :, ::-1].transpose(0, 3, 1, 2))
            parts.append(model(torch.from_numpy(rgb).to(device).float()).float().cpu().numpy())
    model.train(was_training)
    return normalize(np.concatenate(parts))


def lfw_verification(embeddings: np.ndarray, first, second, same) -> tuple[float, float, float]:
    """Returns (mean accuracy, its spread over the 10 folds, mean chosen threshold)."""
    similarities = np.sum(embeddings[first] * embeddings[second], axis=1)
    folds = np.arange(len(similarities)) // 600
    thresholds = np.arange(-1.0, 1.0, 0.001)
    accuracies, chosen = [], []
    for fold in range(10):
        tune = folds != fold
        tune_accuracy = ((similarities[tune][None, :] >= thresholds[:, None]) == same[tune][None, :]).mean(axis=1)
        threshold = thresholds[tune_accuracy.argmax()]
        accuracies.append(((similarities[~tune] >= threshold) == same[~tune]).mean())
        chosen.append(threshold)
    return float(np.mean(accuracies)), float(np.std(accuracies)), float(np.mean(chosen))


def identification_test(
    embeddings, names, thresholds, photos_per_person=5, registered_count=100, stranger_count=1000, draws=(1, 2, 3)
):
    """Returns, for each threshold, (share of registered people's photos named correctly,
    share of strangers given a name), averaged over several random draws of people."""
    photos_of = defaultdict(list)
    for number, name in enumerate(names):
        photos_of[name].append(number)
    candidates = sorted(name for name, photos in photos_of.items() if len(photos) >= 6)

    correct = np.zeros(len(thresholds))
    named = np.zeros(len(thresholds))
    for seed in draws:
        rng = np.random.default_rng(seed)
        registered = rng.choice(candidates, registered_count, replace=False)
        database, tests, test_labels = [], [], []
        for label, name in enumerate(registered):
            photos = rng.permutation(photos_of[name])
            average = embeddings[photos[:photos_per_person]].mean(axis=0)
            database.append(average / np.linalg.norm(average))
            tests += list(photos[5:10])  # the same test photos whatever photos_per_person is
            test_labels += [label] * len(photos[5:10])
        others = sorted(set(photos_of) - set(registered))
        strangers = [rng.choice(photos_of[name]) for name in rng.choice(others, stranger_count, replace=False)]

        database = np.array(database)
        test_similarities = embeddings[tests] @ database.T
        best_person, best_similarity = test_similarities.argmax(axis=1), test_similarities.max(axis=1)
        stranger_best = (embeddings[strangers] @ database.T).max(axis=1)
        for number, threshold in enumerate(thresholds):
            correct[number] += np.mean((best_person == np.array(test_labels)) & (best_similarity >= threshold))
            named[number] += np.mean(stranger_best >= threshold)
    return correct / len(draws), named / len(draws)


def choose_threshold(embeddings, names) -> float:
    """The lowest threshold (to 0.01) at which no more than 0.4% of strangers get a name."""
    thresholds = np.round(np.arange(0.20, 0.80, 0.01), 2)
    _, named = identification_test(embeddings, names, thresholds)
    return float(thresholds[np.argmax(named <= STRANGERS_ALLOWED)])


def report(title: str, embeddings: np.ndarray, names, stems) -> None:
    accuracy, spread, threshold = lfw_verification(embeddings, *load_lfw_pairs(stems))
    print(f"\n===== {title}")
    print(f"Standard LFW test (6,000 pairs): {accuracy:.2%} correct (+/- {spread:.2%}), best threshold about {threshold:.2f}")

    chosen = choose_threshold(embeddings, names)
    print(f"Lowest threshold giving at most {STRANGERS_ALLOWED:.1%} of strangers a name: {chosen:.2f}")
    print("College test, 100 registered people:")
    print("| Photos per person | Threshold | Registered people named correctly | Strangers given a name |")
    print("| --- | --- | --- | --- |")
    for photos_per_person in (5, 3, 1):
        thresholds = sorted({0.363, 0.40, 0.46, 0.50, chosen}) if photos_per_person == 5 else [chosen]
        correct, named = identification_test(embeddings, names, thresholds, photos_per_person)
        for threshold, c, n in zip(thresholds, correct, named):
            print(f"| {photos_per_person} | {threshold:.3g} | {c:.1%} | {n:.1%} |")


def cached_opencv_embeddings(model_file: Path, crops: np.ndarray, mirror: bool = False) -> np.ndarray:
    """LFW embeddings as the app makes them, saved so the next run is quick. With mirror=True,
    each face's embedding is averaged with its mirror image's, like the app's mirror trick."""
    parts = [("", crops)]
    if mirror:
        parts.append(("_mirrored", np.ascontiguousarray(crops[:, :, ::-1])))  # left-right flip
    total = 0
    for suffix, faces in parts:
        cache = DATASETS_DIR / f"lfw_embeddings_{model_file.stem}{suffix}.npy"
        if cache.exists() and cache.stat().st_mtime > model_file.stat().st_mtime:
            embeddings = np.load(cache)
        else:
            print(f"Making LFW{suffix.replace('_', ' ')} embeddings with {model_file.name} (about a minute)...")
            embeddings = embed_with_opencv(model_file, faces)
            np.save(cache, embeddings)
        total = total + embeddings
    return normalize(total)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--mirror", choices=["app", "on", "off"], default="app",
                        help="the mirror trick: on for our model as in app/config.py, off for OpenCV's (default), or on/off for both")
    parser.add_argument("--compare-opencv", action="store_true",
                        help="also test OpenCV's pretrained SFace the same way (downloaded for this, 37 MB)")
    args = parser.parse_args()
    crops, names, stems = load_lfw()
    models = [("Our own SFace", config.OUR_MODEL_FILE, config.MIRROR_TRICK)]
    if args.compare_opencv:
        download_model(OPENCV_SFACE_URL, OPENCV_SFACE_FILE)
        models.insert(0, ("OpenCV's SFace, for comparison", OPENCV_SFACE_FILE, False))
    for title, model_file, mirror_as_in_app in models:
        if not model_file.exists():
            print(f"\n{model_file.name} does not exist yet: train and export our model first.")
            continue
        mirror = mirror_as_in_app if args.mirror == "app" else args.mirror == "on"
        title += ", with the mirror trick" if mirror else ""
        report(title, cached_opencv_embeddings(model_file, crops, mirror), names, stems)


In [ ]:
%%writefile training/train.py
"""Training our own SFace.  Run from the project folder:
    python -m training.train              start, or carry on from the last finished epoch
    python -m training.train --steps 300  a quick speed test; saves nothing

One pass over all 490,623 training photos is an "epoch". After every epoch the model is tested on
the 6,000 LFW pairs and saved to training/runs/<run>/last.pt, so stopping (Ctrl+C, a restart)
loses at most one epoch. When all epochs are done, the finished network is saved as model.pt.

Settings come from the official SFace code (MobileFaceNet, SFace loss, plain SGD with momentum,
learning rate 0.1 divided by 10 at epochs 50, 70 and 80, weight decay 5e-4, batch 256). Changes to
fit a small GPU budget: 90 epochs instead of 125, a short warm-up, and 16-bit arithmetic (see CLAUDE.md).
The first model (run "mobilefacenet_sface") was trained with --epochs 30 --lr-drops 16 24 28 and
without the webcam-style photo changes (training/datasets.py).
The same file runs on this laptop and on Kaggle (training/our_sface_kaggle.ipynb).
"""

import argparse
import csv
import os
import time

import torch

from app.config import PROJECT_DIR
from training.datasets import LFW_ALIGNED, LFW_PAIRS, WebFaceDataset, load_lfw_pairs
from training.evaluate import embed_with_torch, lfw_verification, load_lfw
from training.mobilefacenet import EMBEDDING_SIZE, MobileFaceNet
from training.sface_loss import SFaceLoss

RUNS_DIR = PROJECT_DIR / "training" / "runs"
# The second model: 90 epochs with webcam-style photo changes. The first one is in runs/mobilefacenet_sface.
DEFAULT_RUN = "mobilefacenet_sface_v2"


def learning_rate(args, epoch: int, step: int) -> float:
    """0.1 at the start, divided by 10 at each epoch in args.lr_drops.
    During the first args.warmup_steps steps it climbs from almost 0 to 0.1, so the big early
    updates don't throw the freshly started network off course."""
    rate = args.lr * 0.1 ** sum(epoch >= drop for drop in args.lr_drops)
    if step < args.warmup_steps:
        rate *= (step + 1) / args.warmup_steps
    return rate


def parameter_groups(*modules):
    """Weight decay (gently shrinking weights towards 0, which stops them from memorizing the
    training photos) only for the big weight tables, not for batch-norm values or PReLU slopes."""
    decay, no_decay = [], []
    for module in modules:
        for parameter in module.parameters():
            (decay if parameter.ndim > 1 else no_decay).append(parameter)
    return decay, no_decay


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--run", default=DEFAULT_RUN, help="folder name in training/runs/")
    # The official code drops the learning rate at epochs 50, 70 and 80 of 125. After 80 it barely
    # changes anything (our first run gained nothing from its last drop), so we stop at 90.
    parser.add_argument("--epochs", type=int, default=90)
    parser.add_argument("--lr-drops", type=int, nargs="+", default=[50, 70, 80])
    parser.add_argument("--no-webcam-changes", action="store_true", help="train on the clean photos only, like the first run")
    parser.add_argument("--lr", type=float, default=0.1)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--warmup-steps", type=int, default=1000)
    parser.add_argument("--workers", type=int, default=min(8, os.cpu_count()), help="processes that read photos")
    parser.add_argument("--steps", type=int, default=0, help="stop after this many steps (speed test)")
    parser.add_argument("--precision", choices=["auto", "bfloat16", "float16"], default="auto")
    parser.add_argument("--max-hours", type=float, default=0,
                        help="stop after the epoch that would run past this many hours (Kaggle stops at 12)")
    args = parser.parse_args()
    session_started = time.time()

    torch.manual_seed(1337)
    torch.backends.cudnn.benchmark = True  # lets cuDNN pick the fastest way to run each layer
    device = torch.device("cuda")

    # 16-bit numbers halve the GPU's work. Newer GPUs (RTX 30xx and later, compute capability 8+)
    # have bfloat16, which has the same range as normal 32-bit numbers. Older ones like Kaggle's T4
    # only have float16, whose small range can round tiny gradients to 0, so the loss is first
    # multiplied by a large number and the gradients divided by it again (the GradScaler).
    if args.precision == "auto":
        args.precision = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
    sixteen_bit = torch.bfloat16 if args.precision == "bfloat16" else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=args.precision == "float16")
    print(f"GPU: {torch.cuda.get_device_name()} (1 of {torch.cuda.device_count()} used), {args.precision}")

    dataset = WebFaceDataset(webcam_changes=not args.no_webcam_changes)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=args.workers,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    # Only one GPU is used, even on Kaggle's "T4 x2". Splitting each batch over both T4s with
    # torch.nn.DataParallel was measured at 286 photos/s against 1,604 on a single T4: for a
    # network this small, copying it to the second GPU every step costs more than it saves.
    # channels_last (storing each pixel's channels together) was faster too: 1,604 against 1,121.
    model = MobileFaceNet().to(device).to(memory_format=torch.channels_last)
    loss_function = SFaceLoss(EMBEDDING_SIZE, dataset.num_people).to(device)
    decay, no_decay = parameter_groups(model, loss_function)
    optimizer = torch.optim.SGD(
        [{"params": decay, "weight_decay": 5e-4}, {"params": no_decay, "weight_decay": 0.0}],
        lr=args.lr,
        momentum=0.9,
    )
    print(f"{len(dataset):,} photos of {dataset.num_people:,} people, "
          f"{sum(p.numel() for p in model.parameters()):,} weights in the network")

    run_dir = RUNS_DIR / args.run
    checkpoint_file = run_dir / "last.pt"
    first_epoch, step = 0, 0
    if checkpoint_file.exists() and not args.steps:
        checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint["model"])
        loss_function.load_state_dict(checkpoint["loss_function"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        first_epoch, step = checkpoint["epoch"] + 1, checkpoint["step"]
        print(f"Carrying on from the end of epoch {first_epoch}")

    has_lfw = LFW_ALIGNED.exists() and LFW_PAIRS.exists()
    if has_lfw:
        lfw_crops, _, lfw_stems = load_lfw()
        lfw_pairs = load_lfw_pairs(lfw_stems)
    else:
        print("No aligned LFW found: the LFW test after each epoch is skipped.")

    for epoch in range(first_epoch, args.epochs):
        model.train()
        started = time.time()
        totals = {"loss": 0.0, "accuracy": 0.0, "batches": 0}
        window_started, window_photos = time.time(), 0
        for faces, labels in loader:
            rate = learning_rate(args, epoch, step)
            for group in optimizer.param_groups:
                group["lr"] = rate

            faces = faces.to(device, non_blocking=True).float().contiguous(memory_format=torch.channels_last)
            labels = labels.to(device, non_blocking=True)
            # The network runs in 16-bit; the loss itself is computed in full 32-bit precision.
            with torch.autocast("cuda", dtype=sixteen_bit):
                embeddings = model(faces)
            loss, cosines = loss_function(embeddings.float(), labels)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()  # without float16 the scaler does nothing
            scaler.step(optimizer)
            scaler.update()

            step += 1
            window_photos += len(labels)
            if step % 200 == 0:
                loss_value = loss.item()
                # "Training accuracy": how often the nearest class centre is the right person.
                accuracy = (cosines.argmax(dim=1) == labels).float().mean().item()
                totals["loss"] += loss_value
                totals["accuracy"] += accuracy
                totals["batches"] += 1
                speed = window_photos / (time.time() - window_started)
                print(f"epoch {epoch + 1} step {step} loss {loss_value:.2f} training accuracy {accuracy:.1%} "
                      f"learning rate {rate:.4f} {speed:.0f} photos/s", flush=True)
                window_started, window_photos = time.time(), 0
            if args.steps and step >= args.steps:
                print("Speed test finished.")
                return

        minutes = (time.time() - started) / 60
        lfw_accuracy = float("nan")
        if has_lfw:
            lfw_accuracy, _, _ = lfw_verification(embed_with_torch(model, lfw_crops), *lfw_pairs)
        print(f"===== epoch {epoch + 1} done in {minutes:.1f} min, LFW accuracy {lfw_accuracy:.2%}", flush=True)

        run_dir.mkdir(parents=True, exist_ok=True)
        temporary = checkpoint_file.with_suffix(".tmp")
        torch.save(
            {"model": model.state_dict(), "loss_function": loss_function.state_dict(),
             "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
             "epoch": epoch, "step": step, "args": vars(args)},
            temporary,
        )
        temporary.replace(checkpoint_file)  # a crash while saving never damages the last good checkpoint
        log_file = run_dir / "log.csv"
        new_log = not log_file.exists()
        with open(log_file, "a", newline="") as file:
            writer = csv.writer(file)
            if new_log:
                writer.writerow(["epoch", "loss", "training_accuracy", "lfw_accuracy", "minutes"])
            batches = totals["batches"]  # only every 200th step is measured; a tiny test set may have none
            writer.writerow([epoch + 1, round(totals["loss"] / batches, 3) if batches else "",
                             round(totals["accuracy"] / batches, 4) if batches else "",
                             round(lfw_accuracy, 4), round(minutes, 1)])

        hours_after_next_epoch = (time.time() - session_started + minutes * 60) / 3600
        if args.max_hours and hours_after_next_epoch > args.max_hours and epoch + 1 < args.epochs:
            print(f"Stopping after epoch {epoch + 1}: another epoch would pass the {args.max_hours} hour limit.")
            break

    torch.save(model.state_dict(), run_dir / "model.pt")
    print(f"Training finished. Network saved to {run_dir / 'model.pt'}")


if __name__ == "__main__":
    main()


Find the training photos (the dataset added with **Add Input**):

In [ ]:
# The packed photos (2.7 GB) go to /tmp, which is not saved with the notebook's output.
os.environ["FACE_DATASETS_DIR"] = "/tmp/datasets"
found = None
for depth in range(1, 6):  # the folder of person folders (id_0, id_1, ...) can sit a few levels down
    found = next((path for path in Path("/kaggle/input").glob("*/" * depth + "id_*") if path.is_dir()), None)
    if found:
        break
assert found, "Add the dataset 'webface-112x112' by yakhyokhuja with 'Add Input' first."
WEBFACE = found.parent
print(WEBFACE)


Pack the training photos into one file, and download and align LFW for the test after each epoch:

In [ ]:
run(f'python -m training.datasets --webface "{WEBFACE}"')


Train. Every epoch prints LFW accuracy and saves `training/runs/mobilefacenet_sface_v2/last.pt`.
Kaggle stops a run after 12 hours, so training ends early (and still saves the model) if it would run past 11.5 hours
from the start of this notebook:

In [ ]:
hours_left = 11.5 - (time.time() - NOTEBOOK_STARTED) / 3600
run(f"python -m training.train --max-hours {hours_left:.2f}")


Results. The downloaded models are deleted so the output only holds what is needed:

In [ ]:
shutil.rmtree("models", ignore_errors=True)
run_dir = Path("training/runs/mobilefacenet_sface_v2")
print((run_dir / "log.csv").read_text())
for path in sorted(run_dir.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")
